Initialize **Objective**

In [32]:
import numpy as np
from modopt import Problem


class Rosenbrock2D(Problem):
    def initialize(self, ):
        # Name your problem
        self.problem_name = 'Rosenbrock2D'

    def setup(self):
        # Add design variables of your problem
        self.add_design_variables('x',
                                  shape=(2, ),
                                  vals=np.array([.3, .3]))
        self.add_objective('f')

    def setup_derivatives(self):
        # Declare objective gradient and its shape
        self.declare_objective_gradient(wrt='x', )

    # Compute the value of the objective with given design variable values
    def compute_objective(self, dvs, obj):
        x, y = dvs['x']
        obj['f'] = (1-x)**2 + 100*(y-x**2)**2

    def compute_objective_gradient(self, dvs, grad):
        x = dvs['x']
        grad['x'] = np.array([
            -400 * x[0] * (x[1] - x[0]**2) + 2 * (x[0] - 1),
            200 * (x[1] - x[0]**2)
        ])

Initialize **Optimizer**

In [62]:
import numpy as np
import time
from modopt import Optimizer
from scipy.stats.qmc import Sobol


class SimpleGA(Optimizer):


    def initialize(self):
        # Name your algorithm
        self.solver_name = 'Genetic_Algorithm'

        self.obj = self.problem._compute_objective
        self.grad = self.problem._compute_objective_gradient

        self.options.declare('maxiter', default=1000, types=int)
        self.options.declare('opt_tol', default=1e-5, types=float)
        self.options.declare('initialPopulationSize', default = 2**9, types=int)
        self.options.declare('rangeLow', default = -4.0, types = float)
        self.options.declare('rangeHigh', default= 4.0, types = float)
        self.options.declare('mutationRate', default = 0.2, types = float)
        self.options.declare('mutationStd', default = 1.0, types = float)

        # Enable user to specify, as a list, which among the available outputs
        # need to be written to output files
        self.options.declare('readable_outputs', types=list, default=[])

        # Specify format of outputs available from your optimizer after each iteration
        self.available_outputs = {
            'itr': int,
            'obj': float,
            # for arrays from each iteration, shapes need to be declared
            'x': (float, (self.problem.nx, )),
            'opt': float,
            'time': float,
        }

    def setup(self):
        # Instantiate any modules you need for the algorithm
        pass

    def Populate(self):
        n = self.options['initialPopulationSize']

        # Generate Sobol sequence in [0,1]
        qrng = Sobol(d=self.problem.nx, scramble=True)
        matrix = qrng.random(n)

        # Scale to [-4,4]
        population = self.options['rangeLow'] + (self.options['rangeHigh']- self.options['rangeLow']) * matrix
        return population  
    
    def Evaluate(self, population):
        return np.array([self.obj(ind) for ind in population])

    def Fitness_Selection(self,population):
        N, D = population.shape 
        Evaluation = self.Evaluate(population)
        
        population_with_scores = np.column_stack((population, Evaluation))
        population_with_scores = population_with_scores[population_with_scores[:, -1].argsort()]
        
        num_elites = N // 2  # Select half the population
        tournament_size = max(2, N // 5)  # Set tournament size

        selected_indices = []
        for _ in range(num_elites):
            tournament_contestants = np.random.choice(N, tournament_size, replace=False)
            best_index = tournament_contestants[np.argmin(Evaluation[tournament_contestants])]
            selected_indices.append(best_index)

        elites = population[selected_indices]

        return elites 

    def Crossover(self,elites):
        np.random.shuffle(elites)
        N = len(elites)

        offspring = []

        for i in range(0,N,2):
            if i + 1 < N:
                parent1, parent2 = elites[i], elites[i+1]

                var_index = np.random.choice([0,1])

                alpha1 = np.random.uniform(0, 1)
                alpha2 = np.random.uniform(0, 1)

                child1, child2 = parent1.copy(), parent2.copy()
                child1[var_index] = alpha1 * parent1[var_index] + (1 - alpha1) * parent2[var_index]
                child2[var_index] = alpha2 * parent1[var_index] + (1 - alpha2) * parent2[var_index]

                offspring.extend([child1, child2])

        return offspring

    def Mutation(self, elites, offspring):
        newPopulation = np.vstack((elites,offspring))

        N, D = newPopulation.shape

        mutation_mask = np.random.rand(N, D) < self.options['mutationRate']

        gaussian_noise = np.random.normal(0, self.options['mutationStd'], (N, D))
        mutatedPopulation = np.copy(newPopulation)
        mutatedPopulation += mutation_mask * gaussian_noise

        mutatedPopulation = np.clip(mutatedPopulation, self.options['rangeLow'], self.options['rangeHigh'])

        return newPopulation

    def solve(self):
        nx = self.problem.nx
        x = self.problem.x0
        opt_tol = self.options['opt_tol']
        maxiter = self.options['maxiter']

        obj = self.obj
        grad = self.grad

        start_time = time.time()

        # Setting intial values for initial iterates
        x_k = x * 1.
        f_k = obj(x_k)
        g_k = grad(x_k)

        # Iteration counter
        itr = 0

        # Optimality
        opt = float('inf')

        # Initializing outputs
        self.update_outputs(itr=0,
                            x=x_k,
                            obj=f_k,
                            opt=opt,
                            time=time.time() - start_time)

        population = self.Populate()

        while (opt > opt_tol and itr < maxiter):
            itr_start = time.time()
            itr += 1
            elites = self.Fitness_Selection(population)

            offspring = self.Crossover(elites)

            population = self.Mutation(elites,offspring)

            final_evaluations = self.Evaluate(population)
            optimal_index = np.argmin(final_evaluations)
            f_k = final_evaluations[optimal_index]
            opt = f_k
            x_k = population[optimal_index]

            # Append arrays inside outputs dict with new values from the current iteration
            self.update_outputs(itr=itr,
                                x=x_k,
                                obj=f_k,
                                opt=opt,
                                time=time.time() - start_time)

        self.total_time = time.time() - start_time

        self.results = {
            'x': x_k,
            'objective': f_k,
            'optimality': opt,
            'itr': itr,
            'time': self.total_time
        }

        # Run post-processing for the Optimizer() base class
        self.run_post_processing()

        return self.results

In [ ]:
import time
from modopt import Optimizer
from deap import base, creator, tools
import random


class DEAPGeneric(Optimizer):


    def initialize(self):

        # Name your algorithm
        self.solver_name = 'DEAP_Genetic_Algorithm'

        self.obj = self.problem._compute_objective
        self.grad = self.problem._compute_objective_gradient

        self.options.declare('maxiter', default=1000, types=int)
        self.options.declare('opt_tol', default=1e-5, types=float)
        self.options.declare('initialPopulationSize', default=200, types=int)
        self.options.declare('rangeLow', default = -4.0, types = float)
        self.options.declare('rangeHigh', default =  4.0, types = float)
        self.options.declare('mutationRateGene', default = 0.1, types = float)
        self.options.declare('alpha', default = 0.5, types = float)
        self.options.declare('tournsize', default = 3, types = int)
        self.options.declare('cxProb' , default =  0.5, types = float)
        self.options.declare('mutationRateInd', default =   0.2, types = float)

        # Enable user to specify, as a list, which among the available outputs
        # need to be written to output files
        self.options.declare('readable_outputs', types=list, default=[])

        # Specify format of outputs available from your optimizer after each iteration
        self.available_outputs = {
            'itr': int,
            'obj': float,
            # for arrays from each iteration, shapes need to be declared
            'x': (float, (self.problem.nx, )),
            'opt': float,
            'time': float,
        }

    def setup(self):
        if not hasattr(creator, "FitnessMin"):
            # -1.0 Weight means minimization, 1.0 for Maximization
            creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
        if not hasattr(creator, "Individual"):
            creator.create("Individual", list, fitness=creator.FitnessMin)

        IND_SIZE = self.problem.nx

        self.toolbox = base.Toolbox()
        self.toolbox.register("attribute", random.uniform, self.options['rangeLow'], self.options['rangeHigh'])  # Adjusted range
        self.toolbox.register("individual", tools.initRepeat, creator.Individual,
                 self.toolbox.attribute, n=IND_SIZE)
        self.toolbox.register("population", tools.initRepeat, list, self.toolbox.individual)

        self.toolbox.register("mate", tools.cxBlend, alpha=self.options['alpha'])
        self.toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=(self.options['rangeHigh'] - self.options['rangeLow'])*0.1, indpb=self.options['mutationRateGene'])
        self.toolbox.register("select", tools.selTournament, tournsize=self.options['tournsize'])
        def modopt_evaluate(individual):
            return (self.problem._compute_objective(individual),)

        self.toolbox.register("evaluate", modopt_evaluate)

    def solve(self):
        x = self.problem.x0
        opt_tol = self.options['opt_tol']
        maxiter = self.options['maxiter']

        obj = self.obj
        grad = self.grad

        start_time = time.time()

        # Setting intial values for initial iterates
        x_k = x * 1.
        f_k = obj(x_k)
        g_k = grad(x_k)

        # Iteration counter
        itr = 0

        # Optimality
        opt = float('inf')

        # Initializing outputs
        self.update_outputs(itr=0,
                            x=x_k,
                            obj=f_k,
                            opt=opt,
                            time=time.time() - start_time)

        pop = self.toolbox.population(n=self.options['initialPopulationSize'])
        CXPB, MUTPB, NGEN = self.options['cxProb'], self.options['mutationRateInd'], maxiter

        # Evaluate the entire population
        for ind in pop:
            ind.fitness.values = self.toolbox.evaluate(ind)

        while (opt > opt_tol and itr < NGEN):
            # Select the next generation individuals
            offspring = self.toolbox.select(pop, len(pop))
            # Clone the selected individuals
            offspring = list(map(self.toolbox.clone, offspring))

            # Apply crossover and mutation on the offspring
            for child1, child2 in zip(offspring[::2], offspring[1::2]):
                if random.random() < CXPB:
                    self.toolbox.mate(child1, child2)
                    del child1.fitness.values
                    del child2.fitness.values

            for mutant in offspring:
                if random.random() < MUTPB:
                    self.toolbox.mutate(mutant)
                    del mutant.fitness.values

            # Evaluate the individuals with an invalid fitness
            invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
            fitnesses = map(self.toolbox.evaluate, invalid_ind)
            for ind, fit in zip(invalid_ind, fitnesses):
                ind.fitness.values = fit

            # The population is entirely replaced by the offspring
            pop[:] = offspring

            # Output the best solution
            if len(self.obj(pop[0])) == 1:
                best_ind = tools.selBest(pop, 1)[0]
            else:
                best_inds = tools.sortNondominated(pop, len(pop), first_front_only=True)[0]
                best_ind = random.choice(best_inds)

            f_k = best_ind.fitness.values[0]
            opt = f_k

            x_k = np.array(best_ind)

            itr += 1
            
            # Append arrays inside outputs dict with new values from the current iteration
            self.update_outputs(itr=itr,
                                x=x_k,
                                obj=f_k,
                                opt=opt,
                                time=time.time() - start_time)

        self.total_time = time.time() - start_time

        self.results = {
            'x': x_k,
            'objective': f_k,
            'optimality': opt,
            'itr': itr,
            'time': self.total_time
        }

        # Run post-processing for the Optimizer() base class
        self.run_post_processing()

        return self.results

In [63]:
# Set your optimality tolerance
opt_tol = 1E-8
# Set maximum optimizer iteration limit
maxiter = 1000

Pop = 700

prob = Rosenbrock2D()

# Set up your optimizer with your problem and pass in optimizer parameters
# And declare outputs to be stored
optimizer = SimpleGA(prob,
                            opt_tol=opt_tol,
                            maxiter=maxiter,
                            readable_outputs=['itr', 'obj', 'x', 'opt', 'time'])

# Check first derivatives at the initial guess, if needed
optimizer.check_first_derivatives(prob.x0)

# Solve your optimization problem
optimizer.solve()

# Print results of optimization (summary_table contains information from each iteration)
optimizer.print_results(summary_table=True)

# Print any output that was declared
# Since the arrays are long, here we only print the last entry and
# verify it with the print_results() above

print('\n')
print(optimizer.results['itr'])
print(optimizer.results['x'])
print(optimizer.results['time'])
print(optimizer.results['objective'])
print(optimizer.results['optimality'])

optimizer = DEAPGeneric(prob,
                            opt_tol=opt_tol,
                            maxiter=maxiter,
                            initialPopulationSize = Pop,
                            readable_outputs=['itr', 'obj', 'x', 'opt', 'time'])


# Solve your optimization problem
optimizer.solve()

# Print results of optimization (summary_table contains information from each iteration)
optimizer.print_results(summary_table=True)

# Print any output that was declared
# Since the arrays are long, here we only print the last entry and
# verify it with the print_results() above

print('\n')
print(optimizer.results['itr'])
print(optimizer.results['x'])
print(optimizer.results['time'])
print(optimizer.results['objective'])
print(optimizer.results['optimality'])

Setting objective name as "f".

----------------------------------------------------------------------------
Derivative type | Calc norm  | FD norm    | Abs error norm | Rel error norm 
----------------------------------------------------------------------------

Gradient        | 4.9715e+01 | 4.9715e+01 | 1.0012e-04     | 2.0140e-06    
----------------------------------------------------------------------------


	Solution from modOpt:
	----------------------------------------------------------------------------------------------------
	Problem                  : Rosenbrock2D
	Solver                   : Genetic_Algorithm
	objective                : 0.0005508149364763918
	optimality               : 0.0005508149364763918
	itr                      : 1000
	time                     : 18.897570371627808
	total_callbacks          : 1024006
	obj_evals                : 1024004
	grad_evals               : 2
	hess_evals               : 0
	con_evals                : 0
	jac_evals                :

DEAP